<a href="https://colab.research.google.com/github/FC-Andrade/Analises-complementares/blob/main/CLUSTERING_ANALYSIS_(Apo_vs_Complex).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# CLUSTERING ANALYSIS - (Apo vs Complex)
# =====================================================
# Requer: MDAnalysis, scikit-learn, numpy, matplotlib, seaborn

import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from MDAnalysis.analysis import align, rms
from sklearn.cluster import DBSCAN

sns.set(style="whitegrid", context="talk")
plt.rcParams['figure.figsize'] = (6, 4)

# --- Carregar as trajetórias ---
u_apo = mda.Universe("apo.tpr", "apo.xtc")
u_c16 = mda.Universe("c16.tpr", "c16.xtc")
selection = "protein and name CA"

# =====================================================
# 1️⃣ Extração das coordenadas (reduzindo frames para eficiência)
# =====================================================
def extract_coords(universe, stride=10):
    atoms = universe.select_atoms(selection)
    coords = np.array([atoms.positions.copy() for ts in universe.trajectory[::stride]])
    return coords

coords_apo = extract_coords(u_apo)
coords_c16 = extract_coords(u_c16)

# =====================================================
# 2️⃣ Cálculo da matriz RMSD
# =====================================================
def rmsd_matrix(coords):
    n = coords.shape[0]
    mat = np.zeros((n, n))
    for i in range(n):
        diff = coords - coords[i]
        mat[i] = np.sqrt(np.mean(np.sum(diff**2, axis=2), axis=1))
    return mat

rmsd_apo = rmsd_matrix(coords_apo)
rmsd_c16 = rmsd_matrix(coords_c16)

# =====================================================
# 3️⃣ Clustering (DBSCAN)
# =====================================================
def run_clustering(rmsd_mat, eps=0.25, min_samples=3):
    clustering = DBSCAN(eps=eps, min_samples=min_samples, metric="precomputed").fit(rmsd_mat)
    labels = clustering.labels_
    unique, counts = np.unique(labels[labels >= 0], return_counts=True)
    cluster_data = sorted(zip(unique, counts), key=lambda x: x[1], reverse=True)
    return labels, cluster_data

labels_apo, clusters_apo = run_clustering(rmsd_apo)
labels_c16, clusters_c16 = run_clustering(rmsd_c16)

# =====================================================
# 4️⃣ Gráficos: distribuição e curva cumulativa
# =====================================================
def plot_cluster_stats(cluster_data, color, title):
    cluster_ids, counts = zip(*cluster_data)
    total = sum(counts)
    cum_percent = np.cumsum(np.array(counts) / total * 100)

    fig, axs = plt.subplots(2, 1, figsize=(4, 5))
    axs[0].bar(cluster_ids, counts, color=color)
    axs[0].set_ylabel("Number of structures")
    axs[0].set_title(title)

    axs[1].plot(cluster_ids, cum_percent, "o-", color=color)
    axs[1].axhline(80, color=color, linestyle="--", alpha=0.7)
    idx_80 = np.argmax(cum_percent >= 80)
    axs[1].axvline(cluster_ids[idx_80], color=color, linestyle="--", alpha=0.7)
    axs[1].set_xlabel("Cluster ID")
    axs[1].set_ylabel("Cumulative (%)")
    plt.tight_layout()
    plt.show()

plot_cluster_stats(clusters_apo, "steelblue", "Apo")
plot_cluster_stats(clusters_c16, "darkorange", "C16/S1")

# =====================================================
# 5️⃣ Exportar centroides representativos (Painel B)
# =====================================================
def save_cluster_centroids(universe, coords, labels, out_prefix, top_n=5):
    unique, counts = np.unique(labels[labels >= 0], return_counts=True)
    sorted_clusters = [c for _, c in sorted(zip(counts, unique), reverse=True)]
    atoms = universe.select_atoms(selection)

    for i, cid in enumerate(sorted_clusters[:top_n]):
        idx = np.where(labels == cid)[0]
        centroid_idx = idx[0]
        atoms.positions = coords[centroid_idx]
        atoms.write(f"{out_prefix}_cluster{i+1}.pdb")

save_cluster_centroids(u_apo, coords_apo, labels_apo, "apo")
save_cluster_centroids(u_c16, coords_c16, labels_c16, "c16")

print("✅ Cluster analysis complete.")
print("Centroid structures saved as 'apo_clusterX.pdb' and 'c16_clusterX.pdb'")